In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from pathlib import Path
ROOT_DIR = Path("/home/yangdejin/nlpcc/nlpcc_task2")

TRAIN_DATA_FILE = ROOT_DIR / "data" / "track2" / "sft_train.jsonl"
DEV_DATA_FILE = ROOT_DIR / "data" / "track2" / "sft_dev.jsonl"

SFT_MODEL_DIR = ROOT_DIR / "outputs" / "track2" / "qwen35_9b" / "sft"
TRACK1_MODEL_DIR = ROOT_DIR / "outputs" / "track1" / "encoder" / "deberta_v2_xxlarge_moco_scl" / "best_macro_f1_model" / "query_encoder_lora"

OUTPUTS_DIR = ROOT_DIR / "outputs" / "track2" / "qwen35_9b" / "ppo"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
TRACK1_BASE_MODEL = "microsoft/deberta-v2-xxlarge"
TRACK2_BASE_MODEL = os.environ.get("TRACK2_BASE_MODEL", "Qwen/Qwen3.5-9B")
PROMPT_MAX_LENGTH = 512
MAX_NEW_TOKENS = 64
TRACK1_MAX_LENGTH = 512
LOAD_IN_4BIT = True
BF16 = True

# 24G 显存下优先提 batch 提速；如果 OOM，把 PPO_BATCH_SIZE 改成 4 或 2。
PPO_BATCH_SIZE = int(os.environ.get("PPO_BATCH_SIZE", "8"))
PPO_MINI_BATCH_SIZE = int(os.environ.get("PPO_MINI_BATCH_SIZE", "1"))
PPO_GRADIENT_ACCUMULATION_STEPS = int(os.environ.get("PPO_GRADIENT_ACCUMULATION_STEPS", "1"))
PPO_EPOCHS = int(os.environ.get("PPO_EPOCHS", "1"))
GENERATION_BATCH_SIZE = int(os.environ.get("GENERATION_BATCH_SIZE", "2"))
# Qwen + SDPA 在 batched padding mask 下可能报 (*bias) contiguous 错误，默认用 eager 更稳。
ATTN_IMPLEMENTATION = os.environ.get("ATTN_IMPLEMENTATION", "eager")
# 24G GPU 上单独 reference model 会再占一份 9B 显存，默认关闭。
USE_SEPARATE_REF_MODEL = os.environ.get("USE_SEPARATE_REF_MODEL", "0") == "1"

In [4]:
import json
import re
VALUE_LABELS = [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance",
]

label2id = {label : i for i, label in enumerate(VALUE_LABELS)}
id2label = {i : label for i, label in enumerate(VALUE_LABELS)}

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def normalize_space(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def get_target_value(prompt):
    value = prompt.split("\n\nTarget value:\n", 1)[1]
    value = value.split("\n\nTarget value definition:\n", 1)[0]
    return normalize_space(value)

train_rows = read_jsonl(TRAIN_DATA_FILE)
ppo_rows = []
for i, row in enumerate(train_rows):
    prompt = row["prompt"]
    value = get_target_value(prompt)
    ppo_rows.append({
        "idx" : i,
        "prompt" : prompt,
        "target_value" : value,
    })

print("train_rows:", len(train_rows))
print("ppo_rows:", len(ppo_rows))
print("sample target:", ppo_rows[0]["target_value"])
print("sample prompt:", ppo_rows[0]["prompt"][:700])

train_rows: 3520
ppo_rows: 3520
sample target: Conformity–interpersonal
sample prompt: You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



#### load tokenizer and dataset

In [5]:
from datasets import Dataset
from transformers import AutoTokenizer
# 加载tokenizer
tokenizer_scr = str(SFT_MODEL_DIR)
ppo_tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_scr, 
    trust_remote_code = True,
    )
if ppo_tokenizer.pad_token is None:
    ppo_tokenizer.pad_token = ppo_tokenizer.eos_token
ppo_tokenizer.padding_side = "left"

# 数据类型转化
ppo_dataset = Dataset.from_list(ppo_rows)

# 数据编码与映射
def tokenizer_ppo_batch(batch):
    encoded = ppo_tokenizer(
        batch["prompt"],
        truncation = True,
        max_length = PROMPT_MAX_LENGTH,
        padding = False,
    )
    batch["input_ids"] = encoded["input_ids"]
    batch["attention_mask"] = encoded["attention_mask"]
    return batch

ppo_dataset = ppo_dataset.map(tokenizer_ppo_batch, batched=True)

print("ppo dataset:", len(ppo_dataset))
print("pad_token:", ppo_tokenizer.pad_token, ppo_tokenizer.pad_token_id)
print("sample input length:", len(ppo_dataset[0]["input_ids"]))
print("sample target:", ppo_dataset[0]["target_value"])
print("sample row prompt:", ppo_dataset[0]["prompt"])
print("sample prompt input_ids:", ppo_dataset[0]["input_ids"])

Map:   0%|          | 0/3520 [00:00<?, ? examples/s]

ppo dataset: 3520
pad_token: <|endoftext|> 248044
sample input length: 105
sample target: Conformity–interpersonal
sample row prompt: You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:

sample prompt input_ids: [2523, 513, 2574, 264, 14611, 11, 264, 3296, 11, 321, 264, 2100, 3611, 869, 13, 19208, 799, 61446, 11, 21716, 1965, 421, 10926, 279, 3296, 11, 17759, 279, 14611, 11, 321, 17185, 5117, 82, 440, 279, 2100, 869, 13, 271, 52217, 25, 198, 2523, 944, 303, 264, 12501, 6047, 1332, 678, 6436, 13418, 84270, 7077, 16981, 13, 271, 14162, 25, 1

In [6]:
# 加载track1的分类模型
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import PeftModel
import torch
import torch.nn as nn


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class Track1CompactClassifier(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        pooled = mean_pooling(outputs.last_hidden_state, attention_mask)
        logits = self.classifier(pooled.to(self.classifier.weight.dtype))
        return SequenceClassifierOutput(logits=logits)


def torch_load_cpu(path):
    try:
        return torch.load(str(path), map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location="cpu")


def load_track1_model(model_dir):
    model_dir = Path(model_dir)
    has_lora_adapter = (model_dir / "adapter_config.json").exists()
    track1_dir = model_dir.parent if has_lora_adapter else model_dir
    track1_tokenizer = AutoTokenizer.from_pretrained(str(track1_dir), use_fast=True, trust_remote_code=True)
    kwargs = dict(
        num_labels=len(VALUE_LABELS),
        id2label=id2label,
        label2id=label2id,
        trust_remote_code=True,
    )
    heads_file = track1_dir / "heads.pt"
    if has_lora_adapter and heads_file.exists():
        heads = torch_load_cpu(heads_file)
        base_model_name = heads.get("model_name", TRACK1_BASE_MODEL)
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            model_dtype = torch.bfloat16
        elif torch.cuda.is_available():
            model_dtype = torch.float16
        else:
            model_dtype = torch.float32
        base_encoder = AutoModel.from_pretrained(
            base_model_name,
            trust_remote_code=True,
            torch_dtype=model_dtype,
        )
        encoder = PeftModel.from_pretrained(base_encoder, str(model_dir))
        hidden_size = getattr(encoder.config, "hidden_size", None)
        if hidden_size is None:
            hidden_size = encoder.base_model.model.config.hidden_size
        classifier = nn.Linear(hidden_size, len(VALUE_LABELS))
        classifier.load_state_dict(heads["classifier"])
        track1_model = Track1CompactClassifier(encoder, classifier)
    elif has_lora_adapter:
        base_model = AutoModelForSequenceClassification.from_pretrained(
            TRACK1_BASE_MODEL,
            **kwargs,
        )
        track1_model = PeftModel.from_pretrained(base_model, str(model_dir))
    else:
        track1_model = AutoModelForSequenceClassification.from_pretrained(
            str(model_dir),
            **kwargs,
        )
    for p in track1_model.parameters():
        p.requires_grad = False
    device = "cuda" if torch.cuda.is_available() else "cpu"
    track1_model.to(device)
    track1_model.eval()
    return track1_model, track1_tokenizer, device

print("loading track1 model.....")
track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)
print("loaded on", track1_device)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


loading track1 model.....


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/778 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v2-xxlarge
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded on cuda


In [7]:
import numpy as np
@torch.no_grad()
def compute_rewards(responses, target_values):
    texts = ["Response: " + normalize_space(r) for r in responses]
    inputs = track1_tokenizer(
        texts,
        max_length = TRACK1_MAX_LENGTH,
        truncation = True,
        padding = True,
        return_tensors = "pt",
    )    
    inputs = {k:v.to(track1_device) for k, v in inputs.items()}
    output = track1_model(**inputs)
    logits = output.logits.float()
    probs = torch.softmax(logits, dim = -1).cpu().numpy()

    rewards = []
    records = []
    
    for response, target_value, prob in zip(responses, target_values, probs):
        target_id = label2id[target_value]
        pred_id = int(prob.argmax())

        target_prob = float(prob[target_id])
        pred_prob = float(prob[pred_id])

        sorted_ids = np.argsort(prob)[: :-1]
        best_other_id = next(i for i in sorted_ids if i != target_id)

        margin = float(prob[target_id] - prob[best_other_id])

        word_count = len(normalize_space(response).strip())
        length_penalty = 0.0
        if word_count < 10:
            length_penalty = 0.05
        if word_count > 70:
            length_penalty = 0.03
        
        raw_reward = target_prob + 0.25 * margin - length_penalty
        reward = 10 * raw_reward
        reward = float(np.clip(reward, -5.0, 5.0))
        rewards.append(torch.tensor(reward, dtype = torch.float32))
        records.append({
            "target_value": target_value,
            "pred_value":id2label[pred_id],
            "response":response,
            "target_prob": target_prob,
            "pred_prob": pred_prob,
            "reward":reward,
        })
    return rewards, records


In [8]:
import sys
import torch
import transformers

def patch_transformers_for_legacy_trl():
    if hasattr(transformers, "top_k_top_p_filtering"):
        return
    try:
        from transformers.generation.utils import top_k_top_p_filtering
    except ImportError:
        def top_k_top_p_filtering(logits, top_k=0, top_p=1.0, filter_value=-float("Inf"), min_tokens_to_keep=1):
            if top_k > 0:
                top_k = min(max(top_k, min_tokens_to_keep), logits.size(-1))
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits = logits.masked_fill(indices_to_remove, filter_value)

            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(torch.nn.functional.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                if min_tokens_to_keep > 1:
                    sorted_indices_to_remove[..., :min_tokens_to_keep] = 0
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(-1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, filter_value)
            return logits

    transformers.top_k_top_p_filtering = top_k_top_p_filtering

patch_transformers_for_legacy_trl()
for module_name in list(sys.modules):
    if module_name == "trl" or module_name.startswith("trl."):
        sys.modules.pop(module_name, None)

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from trl import AutoModelForCausalLMWithValueHead
import gc

def build_ppo_model_kwargs():
    compute_dtype = torch.bfloat16 if BF16 else torch.float16
    model_kwargs = {
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
        "attn_implementation": ATTN_IMPLEMENTATION,
    }
    if LOAD_IN_4BIT:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"
    else:
        model_kwargs["torch_dtype"] = compute_dtype
        model_kwargs["device_map"] = "auto"
    return model_kwargs

def prepare_ppo_base_model():
    base_model = AutoModelForCausalLM.from_pretrained(
        TRACK2_BASE_MODEL,
        **build_ppo_model_kwargs(),
    )
    if base_model.get_input_embeddings().num_embeddings != len(ppo_tokenizer):
        base_model.resize_token_embeddings(len(ppo_tokenizer))
    base_model.config.pad_token_id = ppo_tokenizer.pad_token_id
    base_model.config.use_cache = False

    if LOAD_IN_4BIT:
        base_model = prepare_model_for_kbit_training(base_model)
    return base_model

def load_ppo_policy_model(model_dir):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    base_model = prepare_ppo_base_model()
    
    policy_model = PeftModel.from_pretrained(
        base_model,
        str(model_dir),
        is_trainable=True,
    )
    policy_model.config.use_cache = False

    policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(policy_model)
    return policy_model
policy_model = load_ppo_policy_model(SFT_MODEL_DIR)
print("policy model loaded")


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

policy model loaded


In [9]:
import gc
def load_ppo_ref_model(model_dir):
    if not USE_SEPARATE_REF_MODEL:
        print("USE_SEPARATE_REF_MODEL=0: skip loading a second 9B reference model to save VRAM.")
        return None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    base_model = prepare_ppo_base_model()

    ref_model = PeftModel.from_pretrained(
        base_model,
        str(model_dir),
        is_trainable=False,
    )
    ref_model.eval()
    ref_model.config.use_cache = False
    ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(ref_model)
    ref_model.eval()

    for p in ref_model.parameters():
        p.requires_grad=False
    return ref_model
ref_model = load_ppo_ref_model(SFT_MODEL_DIR)
print("reference model:", "disabled" if ref_model is None else "loaded")

USE_SEPARATE_REF_MODEL=0: skip loading a second 9B reference model to save VRAM.
reference model: disabled


In [10]:
import inspect
import torch
import transformers

def patch_transformers_for_legacy_trl():
    if hasattr(transformers, "top_k_top_p_filtering"):
        return
    try:
        from transformers.generation.utils import top_k_top_p_filtering
    except ImportError:
        def top_k_top_p_filtering(logits, top_k=0, top_p=1.0, filter_value=-float("Inf"), min_tokens_to_keep=1):
            if top_k > 0:
                top_k = min(max(top_k, min_tokens_to_keep), logits.size(-1))
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits = logits.masked_fill(indices_to_remove, filter_value)

            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(torch.nn.functional.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                if min_tokens_to_keep > 1:
                    sorted_indices_to_remove[..., :min_tokens_to_keep] = 0
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(-1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, filter_value)
            return logits

    transformers.top_k_top_p_filtering = top_k_top_p_filtering

patch_transformers_for_legacy_trl()

from trl import PPOConfig, PPOTrainer

def assert_legacy_ppo_trainer():
    params = inspect.signature(PPOTrainer).parameters
    required = {
        name for name, param in params.items()
        if param.default is inspect._empty
        and param.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)
        and name != "self"
    }
    if {"reward_model", "value_model"} & required:
        raise RuntimeError(
            "当前环境安装的是新版 TRL PPOTrainer，需要 reward_model/value_model；"
            "本 notebook 使用的是旧版 TRL 的 AutoModelForCausalLMWithValueHead + ppo_trainer.step(...) 写法。\n"
            "请在服务器执行：pip install 'trl==0.7.11'，然后重启 kernel 从头运行。"
        )

assert_legacy_ppo_trainer()

def ppo_collator(features):
    batch = {}
    for key in features[0].keys():
        values = [feature[key] for feature in features]
        if key in ["input_ids", "attention_mask"]:
            values = [torch.tensor(v, dtype=torch.long) for v in values]
        
        batch[key] = values
    return batch

def build_ppo_config():
    backward_batch_size = PPO_MINI_BATCH_SIZE * PPO_GRADIENT_ACCUMULATION_STEPS
    if PPO_BATCH_SIZE % backward_batch_size != 0:
        raise ValueError(
            f"PPO_BATCH_SIZE={PPO_BATCH_SIZE} must be a multiple of "
            f"PPO_MINI_BATCH_SIZE * PPO_GRADIENT_ACCUMULATION_STEPS={backward_batch_size}"
        )

    params = inspect.signature(PPOConfig).parameters
    kwargs = {
        "model_name": str(SFT_MODEL_DIR),
        "learning_rate": 1e-6,
        "batch_size": PPO_BATCH_SIZE,
        "mini_batch_size": PPO_MINI_BATCH_SIZE,
        "gradient_accumulation_steps": PPO_GRADIENT_ACCUMULATION_STEPS,
        "remove_unused_columns": False,
        "target_kl": 0.1,
        "log_with": None,
    }
    if "num_ppo_epochs" in params:
        kwargs["num_ppo_epochs"] = PPO_EPOCHS
    elif "ppo_epochs" in params:
        kwargs["ppo_epochs"] = PPO_EPOCHS

    supported_kwargs = {key: value for key, value in kwargs.items() if key in params}
    skipped_kwargs = sorted(set(kwargs) - set(supported_kwargs))
    if skipped_kwargs:
        print("PPOConfig skipped unsupported args:", skipped_kwargs)
    print(
        "PPO speed config:",
        f"batch_size={PPO_BATCH_SIZE},",
        f"mini_batch_size={PPO_MINI_BATCH_SIZE},",
        f"gradient_accumulation_steps={PPO_GRADIENT_ACCUMULATION_STEPS},",
        f"ppo_epochs={PPO_EPOCHS}",
    )
    return PPOConfig(**supported_kwargs)

ppo_config = build_ppo_config()
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=ref_model,
    tokenizer=ppo_tokenizer,
    dataset=ppo_dataset,
    data_collator=ppo_collator,
)
print("num batches:", len(ppo_trainer.dataloader))

PPO speed config: batch_size=8, mini_batch_size=1, gradient_accumulation_steps=1, ppo_epochs=1
num batches: 440


#### begin training

In [11]:
from tqdm.auto import tqdm
generation_kwargs = {
    "max_new_tokens":MAX_NEW_TOKENS,
    "do_sample" : True,
    "temperature" : 0.7,
    "top_p" : 0.9,
    "use_cache" : True,
    "pad_token_id" : ppo_tokenizer.pad_token_id,
    "eos_token_id" : ppo_tokenizer.eos_token_id,
}

def iter_wrapped_models(model):
    seen = set()
    stack = [model]
    while stack:
        obj = stack.pop()
        if obj is None or id(obj) in seen:
            continue
        seen.add(id(obj))
        yield obj
        for attr in ["module", "pretrained_model", "base_model", "model"]:
            stack.append(getattr(obj, attr, None))

def set_gradient_checkpointing(model, enabled):
    method_name = "gradient_checkpointing_enable" if enabled else "gradient_checkpointing_disable"
    for obj in iter_wrapped_models(model):
        method = getattr(obj, method_name, None)
        if callable(method):
            try:
                method()
            except TypeError:
                pass

def set_use_cache(model, enabled):
    for obj in iter_wrapped_models(model):
        config = getattr(obj, "config", None)
        if config is not None and hasattr(config, "use_cache"):
            config.use_cache = enabled

print("generation batch_size:", GENERATION_BATCH_SIZE)
for step, batch in enumerate(tqdm(ppo_trainer.dataloader, desc="ppo training")):
    query_tensors = batch["input_ids"]
    if "target_value" not in batch:
        raise KeyError(f"target_value missing from PPO batch. Available keys: {list(batch.keys())}")
    target_values = batch["target_value"]

    policy_model.eval()
    set_gradient_checkpointing(policy_model, False)
    set_use_cache(policy_model, True)
    try:
        response_tensors = ppo_trainer.generate(
            query_tensors, 
            batch_size = GENERATION_BATCH_SIZE,
            return_prompt = False,
            **generation_kwargs,
        )
    finally:
        set_use_cache(policy_model, False)
        set_gradient_checkpointing(policy_model, True)
        policy_model.train()
    responses = ppo_tokenizer.batch_decode(
        response_tensors,
        skip_special_tokens = True,
    )
    response = [normalize_space(r) for r in responses]
    rewards, reward_records = compute_rewards(
        responses = responses,
        target_values = target_values,
    )
    stats = ppo_trainer.step(
        query_tensors,
        response_tensors,
        rewards,
    )
    reward_num = [value.item() for value in rewards]
    mean_reward = float(np.mean(reward_num))
    if step % 10 == 0:
        print("=" * 80)
        print("step:", step)
        print("mean reward:", round(mean_reward, 4))
        record = reward_records[0]
        print("target:", record["target_value"])
        print("pred:", record["pred_value"])
        print("target_prob:", round(record["target_prob"], 4))
        print("reward:", round(record["reward"], 4))
        print("response:", record["response"])

policy_model.save_pretrained(str(OUTPUTS_DIR))
ppo_tokenizer.save_pretrained(str(OUTPUTS_DIR))

print("saved PPO model to:", OUTPUTS_DIR)


generation batch_size: 2


ppo training:   0%|          | 0/440 [00:00<?, ?it/s]

step: 0
mean reward: 3.1436
target: Conformity–interpersonal
pred: Conformity–interpersonal
target_prob: 0.9987
reward: 5.0
response: I would lend the jacket to my friend, reassuring them that I trust them, and avoid bringing up the risk of damage to prevent any conflict.
step: 10
mean reward: 5.0
target: Conformity–interpersonal
pred: Conformity–interpersonal
target_prob: 0.9928
reward: 5.0
response: I would privately apologize to the colleague, acknowledge their contribution, and reassure them that their input is valued to avoid damaging our relationship.
step: 20
mean reward: 3.0512
target: Universalism–tolerance
pred: Universalism–tolerance
target_prob: 0.8784
reward: 5.0
response: I would prioritize the inclusive experience by finding alternative solutions that respect everyone's comfort, ensuring the event remains accessible to all cultural groups.
step: 30
mean reward: 5.0
target: Conformity–rules
pred: Conformity–rules
target_prob: 0.9996
reward: 5.0
response: I would insist on

/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -1.91 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -3.71 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -11.63 - this might

step: 250
mean reward: 4.0253
target: Security–personal
pred: Security–personal
target_prob: 0.9991
reward: 5.0
response: I will take full responsibility for my own safety and will not expect support from others, believing that only self-discipline can ensure security.


...

step: 420
mean reward: 1.5575
target: Universalism–concern
pred: Benevolence–caring
target_prob: 0.0001
reward: -2.7399
response: I would arrange for the student to become protected from the peer bullying and I will make them to feel safe within the classroom environment.




/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -557.59 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1212: UserWarning: The average ratio of batch (16.69) exceeds threshold 10.00. Skipping batch.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1212: UserWarning: The average ratio of batch (30.46) exceeds threshold 10.00. Skipping batch.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1212: UserWarning: The average ratio of batch (12.52) exceeds threshold 10.00. Sk

step: 430
mean reward: 3.6931
target: Stimulation
pred: Stimulation
target_prob: 0.9967
reward: 5.0
response: I would take a decision to seek an alternative work routine or find new problems to solve to gain a more fresh sensation with my daily lives.user


/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -381.16 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -488.27 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/trl/trainer/ppo_trainer.py:1279: UserWarning: KL divergence is starting to become negative: -517.29 - this 

saved PPO model to: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/ppo
